In [1]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import transforms

from glob import glob
import copy
import numpy as np
from PIL import Image

In [2]:
layer = nn.Conv2d(1, 3, kernel_size=5, stride=1, padding=1)

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

weight torch.Size([3, 1, 5, 5])
bias torch.Size([3])


In [3]:
layer = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1)

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

weight torch.Size([8, 3, 5, 5])
bias torch.Size([8])


In [4]:
layer = nn.Sequential(
    nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))

for name, param in layer.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

0.weight torch.Size([8, 3, 5, 5])
0.bias torch.Size([8])


In [5]:
layer = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=1)
dummy_img = torch.randn(3, 50, 40)

output = layer(dummy_img)

print(output.size())

torch.Size([8, 48, 38])


In [6]:
layer = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
dummy_img = torch.randn(3, 50, 40)

output = layer(dummy_img)

print(output.size())

torch.Size([32, 25, 20])


veri seti: https://www.kaggle.com/datasets/samuelcortinhas/cats-and-dogs-image-classification


In [7]:
train_dir_cats = "../kedi_kopek/train/cats/*"
train_dir_dogs = "../kedi_kopek/train/dogs/*"

train_dict = {"cats": [], "dogs": []}
for cat_file in glob(train_dir_cats):
    train_dict["cats"].append(cat_file)

for dog_file in glob(train_dir_dogs):
    train_dict["dogs"].append(dog_file)

test_dir_cats = "../kedi_kopek/test/cats/*"
test_dir_dogs = "../kedi_kopek/test/dogs/*"

test_dict = {"cats": [], "dogs": []}
for cat_file in glob(test_dir_cats):
    test_dict["cats"].append(cat_file)

for dog_file in glob(test_dir_dogs):
    test_dict["dogs"].append(dog_file)

In [8]:
file_paths = []
labels = np.zeros(len(train_dict["cats"]) + len(train_dict["dogs"]))
labels[len(train_dict["cats"]):] = 1
file_paths.extend(train_dict["cats"])
file_paths.extend(train_dict["dogs"])
for i in [0, 200, 400, 500]:
    print(file_paths[i], labels[i])

../kedi_kopek/train/cats/cat_170.jpg 0.0
../kedi_kopek/train/cats/cat_543.jpg 0.0
../kedi_kopek/train/dogs/dog_232.jpg 1.0
../kedi_kopek/train/dogs/dog_368.jpg 1.0


In [9]:
class CatsDogsDataset(Dataset):
    def __init__(self, files_dict):
        self.file_paths = []
        self.labels = np.zeros(
            len(train_dict["cats"]) + len(train_dict["dogs"]))
        self.labels[len(files_dict["cats"]):] = 1
        self.file_paths.extend(files_dict["cats"])
        self.file_paths.extend(files_dict["dogs"])
        self.transforms = transforms.Compose([
            transforms.Resize(144),
            transforms.RandomCrop(128),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, index):
        img = Image.open(self.file_paths[index])
        img = img.convert("RGB")
        img = self.transforms(img)
        label = labels[index]
        return img.numpy().astype("float32"), label.astype("long")

In [10]:
train_dataset = CatsDogsDataset(train_dict)
BATCH_SIZE = 16
train_data_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True)

In [11]:
test_dataset = CatsDogsDataset(test_dict)
test_data_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False)

In [12]:
batch = next(iter(train_data_loader))[0]
print("Batch Shape:  ", batch.shape)

layer1 = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
x = layer1(batch)
print("After Layer 1:", x.shape)

layer2 = nn.Sequential(
    nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2))
x = layer2(x)
print("After Layer 2:", x.shape)

Batch Shape:   torch.Size([16, 3, 128, 128])
After Layer 1: torch.Size([16, 32, 64, 64])
After Layer 2: torch.Size([16, 64, 32, 32])


In [13]:
# NVIDIA GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
class CNN_1(nn.Module):

    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2))
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 2))

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.fc(x)
        return x

In [16]:
def train_model(model, device):
    model.to(device)

    num_batches = len(train_data_loader)
    num_test_batches = len(test_data_loader)
    PATIENCE = 5
    best_loss = float("inf")
    no_improve = 0
    best_weights = None

    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=0.001)

    for epoch in range(100):
        model.train()
        avg_loss = 0

        for X, Y in train_data_loader:
            X = X.to(device)
            Y = Y.to(device)

            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, Y)
            loss.backward()
            optimizer.step()

            avg_loss += loss.item()

        avg_loss /= num_batches

        model.eval()
        test_loss = 0
        with torch.no_grad():
            for X, Y in test_data_loader:
                X = X.to(device)
                Y = Y.to(device)
                test_loss += criterion(model(X), Y).item()
        test_loss /= num_test_batches

        if test_loss < best_loss:
            best_loss = test_loss
            best_weights = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        print(
            f"Epoch: {epoch+1}, Train Loss: {avg_loss:.4f}, Test Loss: {test_loss:.4f}, No Improve: {no_improve}/{PATIENCE}")

        if no_improve >= PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_weights)
    print(f"Best model restored (test loss: {best_loss:.4f})")

In [17]:
model_1 = CNN_1()
train_model(model_1, device)

Epoch: 1, Train Loss: 0.8964, Test Loss: 0.7887, No Improve: 0/5
Epoch: 2, Train Loss: 0.6715, Test Loss: 0.7346, No Improve: 0/5
Epoch: 3, Train Loss: 0.6604, Test Loss: 0.9923, No Improve: 1/5
Epoch: 4, Train Loss: 0.6476, Test Loss: 1.0035, No Improve: 2/5
Epoch: 5, Train Loss: 0.6686, Test Loss: 0.8885, No Improve: 3/5
Epoch: 6, Train Loss: 0.6260, Test Loss: 0.5914, No Improve: 0/5
Epoch: 7, Train Loss: 0.6485, Test Loss: 0.9312, No Improve: 1/5
Epoch: 8, Train Loss: 0.6172, Test Loss: 0.7762, No Improve: 2/5
Epoch: 9, Train Loss: 0.6259, Test Loss: 0.9280, No Improve: 3/5
Epoch: 10, Train Loss: 0.6263, Test Loss: 0.8376, No Improve: 4/5
Epoch: 11, Train Loss: 0.6121, Test Loss: 0.9732, No Improve: 5/5
Early stopping triggered.
Best model restored (test loss: 0.5914)


In [18]:
model_1.eval()

total, correct = 0, 0
with torch.no_grad():
    for X, Y in test_data_loader:
        X, Y = X.to(device), Y.to(device)
        preds = model_1(X).argmax(dim=1)
        correct += (preds == Y).sum().item()
        total += len(Y)

print(f"{correct}/{total} ({100*correct/total:.1f}%)")

110/140 (78.6%)


In [27]:
batch = next(iter(train_data_loader))[0]
print(batch.shape)

conv_layer_1 = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2))
x = conv_layer_1(batch)
print("L1:", x.shape)

conv_layer_2 = nn.Sequential(
    nn.Conv2d(32, 64, 3, padding=1),
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(2))
x = conv_layer_2(x)
print("L2:", x.shape)

conv_layer_3 = nn.Sequential(
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.MaxPool2d(2))
x = conv_layer_3(x)
print("L3:", x.shape)

torch.Size([16, 3, 128, 128])
L1: torch.Size([16, 32, 64, 64])
L2: torch.Size([16, 64, 32, 32])
L3: torch.Size([16, 128, 16, 16])


In [24]:
class CNN_2(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
                nn.MaxPool2d(2))

        self.features = nn.Sequential(
            conv_block(3, 32),      # 128 → 64
            conv_block(32, 64),     # 64 → 32
            conv_block(64, 128),    # 32 → 16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [25]:
model_2 = CNN_2()
train_model(model_2, device)

Epoch: 1, Train Loss: 3.2698, Test Loss: 1.2211, No Improve: 0/5
Epoch: 2, Train Loss: 0.8242, Test Loss: 0.7059, No Improve: 0/5
Epoch: 3, Train Loss: 0.8712, Test Loss: 1.0639, No Improve: 1/5
Epoch: 4, Train Loss: 0.7531, Test Loss: 0.4279, No Improve: 0/5
Epoch: 5, Train Loss: 0.7008, Test Loss: 2.0799, No Improve: 1/5
Epoch: 6, Train Loss: 0.6941, Test Loss: 0.7566, No Improve: 2/5
Epoch: 7, Train Loss: 0.6704, Test Loss: 0.5150, No Improve: 3/5
Epoch: 8, Train Loss: 0.6186, Test Loss: 0.9892, No Improve: 4/5
Epoch: 9, Train Loss: 0.5978, Test Loss: 1.6153, No Improve: 5/5
Early stopping triggered.
Best model restored (test loss: 0.4279)


In [26]:
model_2.eval()

total, correct = 0, 0
with torch.no_grad():
    for X, Y in test_data_loader:
        X, Y = X.to(device), Y.to(device)
        preds = model_2(X).argmax(dim=1)
        correct += (preds == Y).sum().item()
        total += len(Y)

print(f"{correct}/{total} ({100*correct/total:.1f}%)")

118/140 (84.3%)
